## Setup Colab and sync


In [12]:
#@title Clone the repo and add API key
!git clone https://github.com/amusali/bert_for_patents.git
import os
# Api key setting
os.environ['patentsview_api_key'] = 'Uch9R1RR.BxaQcxV8HK4nBMgTTjX1F9CQW8PF4nXw'  # Replace with your actual key

Cloning into 'bert_for_patents'...
remote: Enumerating objects: 879, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (12/12), done.
^C


In [13]:
#@title Pull latest changes from repo
%cd "/content/bert_for_patents"
!git pull origin master
#@title Add paths to the system and change directory
import sys
sys.path.append('/content/bert_for_patents/05 Analysis/01 Main')
os.chdir("/content/bert_for_patents/05 Analysis/01 Main")

/content/bert_for_patents
From https://github.com/amusali/bert_for_patents
 * branch            master     -> FETCH_HEAD
Already up to date.


In [14]:
#@title Run setup file to get environment
!python "/content/bert_for_patents/05 Analysis/01 Main/setup_colab.py"

import sys
sys.path.append('/content/bert_for_patents/05 Analysis/01 Main')
os.chdir("/content/bert_for_patents/05 Analysis/01 Main")

/content/bert_for_patents/05 Analysis/01 Main/setup_colab.py:10: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources
absl-py==2.1.0 is already installed.
asttokens==2.4.1 is already installed.
astunparse==1.6.3 is already installed.
attrs==23.2.0 is already installed.
beautifulsoup4==4.12.3 is already installed.
bert==2.2.0 is already installed.
bert-tensorflow==1.0.4 is already installed.
certifi==2024.6.2 is already installed.
cffi==1.16.0 is already installed.
charset-normalizer==3.3.2 is already installed.
colorama==0.4.6 is already installed.
comm==0.2.2 is already installed.
contourpy==1.2.1 is already installed.
cycler==0.12.1 is already installed.
debugpy==1.8.1 is already installed.
decorator==5.1.1 is already installed.
dill==0.3.9 is already installed.
erlastic==2.0.0 is already installed.
et-xmlfile==1.1.0 is already installed.
executing==2.0.1 is already installed.
fake-useragent==1.5

In [16]:
#@title Mount Drive to get BERT model
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


# Load and process raw Patent ID and Current CPC data



In [22]:
#@title util

#@title Define classes and functions
import time
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import os
import dill as pickle
from datetime import datetime
import numpy as np

def save(file, folder):
    """
    Saves the embeddings DataFrame to a pickle file with a timestamped filename.
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_name = f"01 CLS embeddings rest - {timestamp}.pkl"
    file_path = os.path.join(folder, file_name)
    with open(file_path, "wb") as f:
        pickle.dump(file, f)
    print(f"Saved embeddings to {file_path}")

def load(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)

def load_tsv(filename):
    """
    Loads a TSV file into a pandas DataFrame.
    """
    return pd.read_csv(filename, sep='\t', low_memory=False)




In [21]:
#@title Load Patent and CPC data

path_patents = r"/content/drive/MyDrive/PhD Data/00 Raw data from PV/g_patent.tsv/g_patent.tsv"
path_cpc = r"/content/drive/MyDrive/PhD Data/00 Raw data from PV/g_cpc_current.tsv/g_cpc_current.tsv"

patents = load_tsv(path_patents)
cpcs = load_tsv(path_cpc)

print(len(patents))
print(len(cpcs))


<ipython-input-17-c66bf71db62d>:31: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(filename, sep='\t')


9075421
55460284


In [96]:
#@title Clean and Combine

# Clean CPC data
#cpcs.head(50)
print(cpcs['cpc_sequence'].value_counts())
cpcs = cpcs[cpcs['cpc_sequence'] == 0]
print(len(cpcs))

# Combine CPC data with Patents
rest = pd.merge(patents, cpcs, on='patent_id', how='outer')
rest = rest[rest['cpc_subclass'].isna()]
print(len(rest))
merged_df = pd.merge(patents, cpcs, on='patent_id', how='inner')
print(len(merged_df))



cpc_sequence
0    8158977
Name: count, dtype: int64
8158977
930578
8144843


In [97]:
#@title Add application date


# Load Applicatoin data
path_applications = r"/content/drive/MyDrive/PhD Data/00 Raw data from PV/g_application.tsv/g_application.tsv"
applications = load_tsv(path_applications)
print(len(applications))

# Combine back to merged df
merged_df['patent_id'] = merged_df['patent_id'].astype(str)
applications['patent_id'] = applications['patent_id'].astype(str)
combined_df = pd.merge(merged_df, applications, on='patent_id', how='inner')
print(len(combined_df))

9073162
8144843


In [98]:


#@title Clean

## Clean unavailable dates that show up as random numbers
combined_df['aux_year'] = combined_df['filing_date'].str.slice(0, 4).astype(int)
combined_df = combined_df[(combined_df['aux_year'] >= 1975) & (combined_df['aux_year'] <= 2025)]
combined_df['application_year'] = pd.to_datetime(combined_df['filing_date']).dt.year
combined_df['grant_year'] = pd.to_datetime(combined_df['patent_date']).dt.year


## Check if grant year > application year
aux = combined_df[combined_df['grant_year'] < combined_df['application_year']]
print(f"Number of patents with grant year before application year is {len(aux)}")
print(f"Total number of patents with CPC aand application dates available is {len(combined_df)}")
assert(len(aux) > 0)

## Drop patents granted before application
combined_df = combined_df[combined_df['grant_year'] >= combined_df['application_year']]

## Keep relevant cols
combined_df = combined_df[['patent_id', 'cpc_subclass', 'application_year', 'grant_year']]
print(len(combined_df))

if combined_df['application_year'].isnull().any():
    print("There are missing values in the column.")
else:
    print("No missing values found in the column.")

Number of patents with grant year before application year is 46
Total number of patents with CPC aand application dates available is 8092997
8092951


In [145]:
#@title Save

print(len(combined_df))

with open("/content/drive/MyDrive/PhD Data/01 CLS Embeddings/Patents with tech fields.pkl", "wb") as f:
        pickle.dump(combined_df, f)


8092951
